# Router Pattern

## Review (4. Tool Calling in LangGraph)

In 4a we built a graph where the LLM decides whether to call a tool or respond directly. The conditional edge (`tools_condition`) handled this binary choice:

- Tool calls present → route to `ToolNode`
- No tool calls → route to `END`

That's already a simple **router** — the LLM routes between two paths.

## Goals

In this notebook we generalise that idea into a full **router pattern** — where the LLM (or a custom function) routes to **multiple** destinations based on the input.

We'll cover:

1. **Simple tool-based router** — Route to different tools via `tools_condition` (recap from 4a)
2. **Custom routing function** — Route to entirely different processing nodes based on classification
3. **Enterprise example** — A customer service router that directs requests to specialised handlers

In [ ]:
%run ../langchain_common.py

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

## Part 1 — Tool-Based Routing (Recap)

This is what we built in notebook 4. The `tools_condition` acts as a router with two branches:

```
START → assistant → tools_condition → tools → assistant (loop)
                                    → END   (no tool call)
```

The LLM itself decides the route by choosing whether or not to call a tool.

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Add a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

tools = [multiply, add]
llm_with_tools = llm_noreason.bind_tools(tools)

def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
tool_graph = builder.compile()

display(Image(tool_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Tool call route
result = tool_graph.invoke({"messages": [HumanMessage(content="What is 6 times 7?")]})
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# Direct response route
result = tool_graph.invoke({"messages": [HumanMessage(content="Hello!")]})
for m in result["messages"]:
    m.pretty_print()

This works, but the routing is limited to "tool call vs. no tool call". What if we want to route to **completely different processing paths** based on the *type* of request?

## Part 2 — Custom Routing with a Classification Node

The router pattern generalises: **use a function (or LLM) to classify the input, then route to the appropriate node**.

Key design idea:
```
START → classifier → conditional_edge → node_A
                                       → node_B
                                       → node_C
```

The classifier can be:
- A **keyword/rule-based** function
- An **LLM call** that returns a structured category
- A **tool call** where each tool represents a different route

Let's build a simple example: route math questions to a calculator node, and everything else to a general chat node.

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

# Extended state to carry the route classification
class RouterState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    route: str  # "math" or "general" — set by the classifier

In [ ]:
def classifier(state: RouterState):
    """Use the LLM to classify the user's request."""
    last_msg = state["messages"][-1].content

    classification = llm_noreason.invoke([
        SystemMessage(content=(
            "Classify the user message into exactly one category.\n"
            "Reply with ONLY the category name, nothing else.\n"
            "Categories: math, general"
        )),
        HumanMessage(content=last_msg),
    ])

    route = classification.content.strip().lower()
    # Normalise to known categories
    if "math" in route:
        route = "math"
    else:
        route = "general"

    print(f"  [classifier] route = {route}")
    return {"route": route}

In [ ]:
def math_node(state: RouterState):
    """Handle math questions using tool-calling LLM."""
    response = llm_with_tools.invoke(state["messages"])

    # If the model wants to call a tool, execute it inline
    if response.tool_calls:
        tool_results = ToolNode(tools).invoke({"messages": state["messages"] + [response]})
        # Get the final answer from the LLM after seeing tool results
        all_msgs = state["messages"] + [response] + tool_results["messages"]
        final = llm_noreason.invoke(all_msgs)
        return {"messages": [response] + tool_results["messages"] + [final]}

    return {"messages": [response]}


def general_node(state: RouterState):
    """Handle general conversation."""
    response = llm_noreason.invoke(
        [SystemMessage(content="You are a helpful, friendly assistant. Keep responses brief.")] +
        state["messages"]
    )
    return {"messages": [response]}

In [ ]:
def route_by_classification(state: RouterState):
    """Conditional edge: read the route from state and return the node name."""
    return state["route"]

# Build the router graph
builder = StateGraph(RouterState)
builder.add_node("classifier", classifier)
builder.add_node("math", math_node)
builder.add_node("general", general_node)

builder.add_edge(START, "classifier")
builder.add_conditional_edges("classifier", route_by_classification, ["math", "general"])
builder.add_edge("math", END)
builder.add_edge("general", END)

router_graph = builder.compile()
display(Image(router_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Test: math question → routes to math node
result = router_graph.invoke({"messages": [HumanMessage(content="What is 12 times 5?")]})
print("\n=== Final Response ===")
result["messages"][-1].pretty_print()

In [ ]:
# Test: general question → routes to general node
result = router_graph.invoke({"messages": [HumanMessage(content="Tell me about the weather in Dublin.")]})
print("\n=== Final Response ===")
result["messages"][-1].pretty_print()

### What Just Happened?

1. The **classifier** node used the LLM to categorise the input as `math` or `general`
2. The result was stored in `state["route"]`
3. The **conditional edge** read that value and routed to the matching node
4. The selected node processed the request independently

This is the **core router pattern**: classify → route → specialise.

## Part 3 — Enterprise Example: Customer Service Router

In real enterprise agents, the router pattern handles domain-specific request types. Let's build a customer service agent that routes between:

| Route | Handler | Description |
|-------|---------|-------------|
| `order_status` | Order lookup node | Checks order status using a tool |
| `technical` | Technical support node | Answers product/technical questions |
| `billing` | Billing node | Handles billing and payment queries |
| `general` | General node | Catches everything else |

In [ ]:
# Enterprise state with route tracking
class CustomerServiceState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    route: str
    customer_id: str

In [ ]:
# Simulated order database
ORDER_DB = {
    "ORD-1234": {"status": "shipped", "eta": "July 8", "item": "Wireless Headphones"},
    "ORD-5678": {"status": "processing", "eta": "July 12", "item": "USB-C Hub"},
    "ORD-9999": {"status": "delivered", "eta": "July 1", "item": "Laptop Stand"},
}

def lookup_order(order_id: str) -> str:
    """Look up the status of a customer order by order ID.

    Args:
        order_id: The order ID to look up (e.g. ORD-1234)
    """
    order = ORDER_DB.get(order_id.upper())
    if order:
        return f"Order {order_id}: {order['item']} — Status: {order['status']}, ETA: {order['eta']}"
    return f"Order {order_id} not found. Please check the order ID."

order_tools = [lookup_order]
llm_with_order_tools = llm_noreason.bind_tools(order_tools)

In [ ]:
CATEGORIES = ["order_status", "technical", "billing", "general"]

def cs_classifier(state: CustomerServiceState):
    """Classify the customer request into a department."""
    last_msg = state["messages"][-1].content

    classification = llm_noreason.invoke([
        SystemMessage(content=(
            "You are a customer service router. Classify the customer message into exactly one category.\n"
            "Reply with ONLY the category name, nothing else.\n"
            f"Categories: {', '.join(CATEGORIES)}\n\n"
            "Guidelines:\n"
            "- order_status: tracking, shipping, delivery, order ID, where is my order\n"
            "- technical: product issues, how-to, troubleshooting, setup, compatibility\n"
            "- billing: payment, refund, invoice, charge, subscription, pricing\n"
            "- general: everything else (greetings, feedback, unrelated)"
        )),
        HumanMessage(content=last_msg),
    ])

    route = classification.content.strip().lower().replace(" ", "_")
    if route not in CATEGORIES:
        route = "general"

    print(f"  [cs_classifier] route = {route}")
    return {"route": route}

In [ ]:
def order_status_node(state: CustomerServiceState):
    """Handle order status queries using the lookup tool."""
    msgs = [
        SystemMessage(content=(
            "You are an order status assistant. Use the lookup_order tool to check order status. "
            "If the customer doesn't provide an order ID, ask for it politely."
        ))
    ] + state["messages"]

    response = llm_with_order_tools.invoke(msgs)

    if response.tool_calls:
        tool_results = ToolNode(order_tools).invoke({"messages": msgs + [response]})
        final = llm_noreason.invoke(msgs + [response] + tool_results["messages"])
        return {"messages": [response] + tool_results["messages"] + [final]}

    return {"messages": [response]}


def technical_node(state: CustomerServiceState):
    """Handle technical support questions."""
    response = llm_noreason.invoke(
        [SystemMessage(content=(
            "You are a technical support specialist. Help the customer with product issues, "
            "troubleshooting, and setup questions. Be concise and helpful."
        ))] + state["messages"]
    )
    return {"messages": [response]}


def billing_node(state: CustomerServiceState):
    """Handle billing and payment questions."""
    response = llm_noreason.invoke(
        [SystemMessage(content=(
            "You are a billing support specialist. Help the customer with payment, refund, "
            "and invoice questions. Be concise. For refunds, always provide a reference number."
        ))] + state["messages"]
    )
    return {"messages": [response]}


def cs_general_node(state: CustomerServiceState):
    """Handle general customer queries."""
    response = llm_noreason.invoke(
        [SystemMessage(content=(
            "You are a friendly customer service representative. Keep responses brief and helpful."
        ))] + state["messages"]
    )
    return {"messages": [response]}

In [ ]:
def cs_route(state: CustomerServiceState):
    return state["route"]

builder = StateGraph(CustomerServiceState)

# Nodes
builder.add_node("classifier", cs_classifier)
builder.add_node("order_status", order_status_node)
builder.add_node("technical", technical_node)
builder.add_node("billing", billing_node)
builder.add_node("general", cs_general_node)

# Edges
builder.add_edge(START, "classifier")
builder.add_conditional_edges("classifier", cs_route, CATEGORIES)
builder.add_edge("order_status", END)
builder.add_edge("technical", END)
builder.add_edge("billing", END)
builder.add_edge("general", END)

cs_graph = builder.compile()
display(Image(cs_graph.get_graph().draw_mermaid_png()))

### Test the Customer Service Router

In [ ]:
# Order status query
result = cs_graph.invoke({
    "messages": [HumanMessage(content="Where is my order ORD-1234?")],
    "customer_id": "C-100",
})
print("\n=== Response ===")
result["messages"][-1].pretty_print()

In [ ]:
# Technical support query
result = cs_graph.invoke({
    "messages": [HumanMessage(content="My headphones won't pair with my laptop via Bluetooth.")],
    "customer_id": "C-200",
})
print("\n=== Response ===")
result["messages"][-1].pretty_print()

In [ ]:
# Billing query
result = cs_graph.invoke({
    "messages": [HumanMessage(content="I was charged twice for my last order. Can I get a refund?")],
    "customer_id": "C-300",
})
print("\n=== Response ===")
result["messages"][-1].pretty_print()

In [ ]:
# General query
result = cs_graph.invoke({
    "messages": [HumanMessage(content="What are your store hours?")],
    "customer_id": "C-400",
})
print("\n=== Response ===")
result["messages"][-1].pretty_print()

## Design Patterns Summary

### When to Use the Router Pattern

| Scenario | Approach |
|----------|----------|
| Binary choice (tool or no tool) | `tools_condition` (Part 1) |
| Few categories, simple logic | LLM classifier + conditional edge (Part 2) |
| Enterprise multi-department routing | LLM classifier with domain prompts (Part 3) |
| Each route needs different tools | Separate tool sets per node |
| Each route needs different system prompts | Specialised nodes with distinct `SystemMessage` |

### Router vs Agent

A **router** is a single-hop: classify → route → respond.

An **agent** adds a loop: the model can call tools, observe results, and decide to act again.

You can combine both — route to a specialised **sub-agent** that has its own tool loop. We'll explore this in the agent notebooks.

## Key Takeaways

| Concept | Details |
|---------|--------|
| `tools_condition` | Built-in binary router: tool call or END |
| Custom state field (`route`) | Store classification in state for the conditional edge to read |
| LLM-as-classifier | Use a focused system prompt to produce a category label |
| `add_conditional_edges(node, fn, [targets])` | Route to one of multiple nodes based on a function |
| Specialised nodes | Each route gets its own system prompt and (optionally) its own tools |
| Enterprise pattern | Classifier → domain-specific handler nodes → END |